## Cross-Country Pay Comparison (USD)

Two dollar views:
- **Market USD** — paycheck converted at the 2023 exchange rate (what they earn globally).
- **PPP USD** — local buying power in dollars (already reflects cost-of-living differences).

PPP factors: USA 1.0, India 22 INR, China 4.1 CNY per PPP USD (IMF 2023).

In [1]:
import sys
from pathlib import Path
ROOT = Path().resolve().parent
sys.path.insert(0, str(ROOT / 'src'))

import pandas as pd
import plotly.express as px

ROLE_LABELS = {
    'software_engineer': 'Software Engineer', 'lawyer': 'Lawyer',
    'physician': 'Physician', 'financial_analyst': 'Financial Analyst',
    'registered_nurse': 'Registered Nurse', 'civil_engineer': 'Civil Engineer',
    'construction_laborer': 'Construction Laborer', 'farm_worker': 'Farm Worker',
    'manufacturing_worker': 'Manufacturing Worker', 'retail_worker': 'Retail Worker',
}

usa = pd.read_csv(ROOT / 'data' / 'processed' / 'merged_usa_data.csv')
india = pd.read_csv(ROOT / 'data' / 'processed' / 'merged_india_data.csv')
china = pd.read_csv(ROOT / 'data' / 'processed' / 'merged_china_data.csv')
df = pd.concat([usa, india, china], ignore_index=True)
df['role_label'] = df['role'].map(ROLE_LABELS)
country_colors = {'USA': '#2171b5', 'India': '#31a354', 'China': '#e6550d'}

In [2]:
# Market USD vs PPP USD for a mid-career SWE — the cost-of-living adjustment made visible
swe = df[(df['role'] == 'software_engineer') & (df['career_stage'] == 'mid')].copy()
compare = pd.melt(
    swe[['country', 'median_salary_usd', 'median_salary_ppp_usd']],
    id_vars='country', value_vars=['median_salary_usd', 'median_salary_ppp_usd'],
    var_name='basis', value_name='salary',
)
compare['basis'] = compare['basis'].map({'median_salary_usd': 'Market USD', 'median_salary_ppp_usd': 'PPP USD (buying power)'})

fig0 = px.bar(
    compare, x='country', y='salary', color='basis', barmode='group',
    category_orders={'country': ['USA', 'India', 'China']},
    color_discrete_map={'Market USD': '#9ecae1', 'PPP USD (buying power)': '#08519c'},
    title='Mid-Career SWE: Market USD vs PPP USD by Country (2023)<br>'
          '<sup>The gap is the cost-of-living adjustment — rupees and yuan stretch further locally</sup>',
    labels={'salary': 'Annual Salary (USD)', 'country': 'Country', 'basis': 'Basis'},
    text='salary',
)
fig0.update_traces(texttemplate='$%{text:,.0f}', textposition='outside')
fig0.show()

In [3]:
swe_all = df[df['role'] == 'software_engineer'].copy()

fig = px.bar(
    swe_all, x='career_stage', y='median_salary_ppp_usd', color='country',
    barmode='group',
    category_orders={'career_stage': ['entry', 'mid', 'senior'], 'country': ['USA', 'India', 'China']},
    color_discrete_map=country_colors,
    title='Software Engineer Salary by Country and Career Stage (PPP-Adjusted USD, 2023)',
    labels={'career_stage': 'Career Stage', 'median_salary_ppp_usd': 'Annual Salary (PPP USD)', 'country': 'Country'},
)
fig.show()

In [4]:
mid = df[df['career_stage'] == 'mid'].copy()
usa_order = (
    mid[mid['country'] == 'USA']
    .sort_values('median_salary_ppp_usd', ascending=False)['role_label'].tolist()
)

fig2 = px.bar(
    mid, x='role_label', y='median_salary_ppp_usd', color='country',
    barmode='group',
    category_orders={'role_label': usa_order, 'country': ['USA', 'India', 'China']},
    color_discrete_map=country_colors,
    title='Mid-Career Salary by Role and Country — PPP-Adjusted USD (2023)',
    labels={'role_label': 'Role', 'median_salary_ppp_usd': 'Annual Salary (PPP USD)', 'country': 'Country'},
)
fig2.update_layout(xaxis_tickangle=-35)
fig2.show()

In [5]:
usa_mid = mid[mid['country'] == 'USA'][['role', 'median_salary_ppp_usd']].rename(
    columns={'median_salary_ppp_usd': 'usa_salary'}
)
pct_df = mid[mid['country'] != 'USA'].merge(usa_mid, on='role')
pct_df['pct_of_usa'] = (pct_df['median_salary_ppp_usd'] / pct_df['usa_salary'] * 100).round(1)
pct_df['role_label'] = pct_df['role'].map(ROLE_LABELS)

fig3 = px.bar(
    pct_df, x='role_label', y='pct_of_usa', color='country',
    barmode='group',
    category_orders={'role_label': usa_order, 'country': ['India', 'China']},
    color_discrete_map={'India': '#31a354', 'China': '#e6550d'},
    title='Mid-Career Salary as % of USA (PPP-Adjusted) — India and China vs USA',
    labels={'role_label': 'Role', 'pct_of_usa': '% of USA salary (PPP)', 'country': 'Country'},
)
fig3.add_hline(y=100, line_dash='dot', line_color='navy', annotation_text='USA baseline')
fig3.update_layout(xaxis_tickangle=-35)
fig3.show()